# Set up a regional MOM6 run within CESM framework (with CrocoDash grid generation)

There are three main sections: 
1. First we create the MOM6 experiment using the [regional-mom6](https://github.com/COSIMA/regional-mom6/) package. This puts everything we need into two repositories, referred to as the `mom_run_dir` and `mom_input_dir`. These contain all of the configuration files (`MOM_input` etc.) and netcdf input files (`hgrid.nc`, OBC segments etc.) respectively. An `experiment` object is also created within the notebook, containing all the information we need about the regional mom6 setup to pass onto CESM.

2. A new MOM6 CESM case is created in the usual way by cloning the CESM repo and running the `create_case` command. 

3. We modify the new CESM case to work with the regional mom6 configuration we prepared earlier. We pass the path to this CESM run and the experiment object to the `setup_cesm` function, which carries out all the required modifications 

This is very much a work in progress! The goal in the short term was to get something working and make it nice later

# SECTION 1: Setup up you MOM6 regional experiment

This follows the normal workflow, copied from the `reanalysis_forced.ipynb` demo of [regional-mom6](https://github.com/COSIMA/regional-mom6/) but modified for a domain around Hawaii. See the documentation of the package for details and documentation

In [1]:
%load_ext autoreload
%autoreload 2
import os
from pathlib import Path
import CrocoDash as crr
import regional_mom6 as rmom6
import CrocoDash as cd
import regional_mom6 as rmom6

## Step 1: Choose our domain, define workspace paths

To make sure that things are working I'd recommend starting with the default example defined below. If this runs ok, then change to a domain of your choice and hopefully it runs ok too! If not, check the [README](https://github.com/COSIMA/regional-mom6/blob/main/README.md) and [documentation](https://regional-mom6.readthedocs.io/) for troubleshooting tips.

You can log in and use [this GUI](https://data.marine.copernicus.eu/product/GLOBAL_MULTIYEAR_PHY_001_030/download) to find the lat/lon of your domain and copy paste below.

In [2]:
expt_name = "gfdl_copy.4.n3b"

latitude_extent = None
longitude_extent = None

date_range = ["1993-01-01 00:00:00", "2020-01-01 00:00:00"]

## Place where all your input files go 
input_dir = Path(f"/glade/u/home/manishrv/documents/nwa12_0.1/mom_input/{expt_name}/")

## Directory where you'll run the experiment from
run_dir = Path(f"/glade/u/home/manishrv/documents/nwa12_0.1/mom_run/{expt_name}/")

## Directory where compiled FRE tools are located (needed for construction of mask tables)
toolpath_dir = Path("")

## Path to where your raw ocean forcing files are stored
glorys_path = os.path.join("/","glade","derecho","scratch","manishrv","gfdl_copy.4.n3b_inputs","glorys" )

## if directories don't exist, create them
for path in (run_dir, glorys_path, input_dir):
    os.makedirs(str(path), exist_ok=True)

## Step 2: Make experiment object
The `regional_mom6.experiment` contains the regional domain basics, and also generates the horizontal and vertical grids, `hgrid` and `vgrid` respectively, and sets up the directory structures. 

In [4]:
expt = rmom6.experiment(
    longitude_extent = longitude_extent,
    latitude_extent = latitude_extent,
    date_range = date_range,
    resolution = None,
    number_vertical_layers = 75,
    layer_thickness_ratio = None,
    depth = 6500,
    minimum_depth = 9.95,
    mom_run_dir = run_dir,
    mom_input_dir = input_dir,
    toolpath_dir = toolpath_dir,
    hgrid_type = "from_file", # This is how we incorporate the grid_gen files
    vgrid_type = "from_file",
    vgrid_path = "/glade/u/home/manishrv/documents/nwa12_0.1/mom_input/gfdl_copy.4.n3b/vgrid.nc",
    expt_name = expt_name,
        boundaries = ["south", "north", "east"],
)

In [15]:
crr.driver.CrocoDashDriver.write_config_file(expt, path = "/glade/u/home/manishrv/documents/nwa12_0.1/mom_run/gfdl_copy.2.n3b/config.json")

2024-12-30 07:19:51,801 - CrocoDash.driver - INFO - Writing Config File.....
2024-12-30 07:19:51,802 - CrocoDash.driver - INFO - Searching /glade/u/home/manishrv/documents/nwa12_0.1/mom_input/gfdl_copy.2.n3b for bathymetry, hgrid, vgrid
2024-12-30 07:19:51,803 - CrocoDash.driver - INFO - Searching /glade/u/home/manishrv/documents/nwa12_0.1/mom_input/gfdl_copy.2.n3b for initial conditions
2024-12-30 07:19:51,804 - CrocoDash.driver - INFO - Searching /glade/u/home/manishrv/documents/nwa12_0.1/mom_input/gfdl_copy.2.n3b for ocean_state conditions
2024-12-30 07:19:51,806 - CrocoDash.driver - INFO - Searching /glade/u/home/manishrv/documents/nwa12_0.1/mom_input/gfdl_copy.2.n3b for tides
2024-12-30 07:19:51,807 - CrocoDash.driver - INFO - Searching /glade/u/home/manishrv/documents/nwa12_0.1/mom_run/gfdl_copy.2.n3b for run files
2024-12-30 07:19:51,811 - CrocoDash.driver - INFO - Done.


{'expt_name': 'gfdl_copy.2.n3b',
 'date_range': ['1993-01-01 00:00:00', '1993-02-01 00:00:00'],
 'latitude_extent': (5.201175132990646, 58.375120887727846),
 'longitude_extent': (-98.74996846773888, -36.33333333333326),
 'resolution': None,
 'number_vertical_layers': 75,
 'layer_thickness_ratio': None,
 'depth': 6500,
 'hgrid_type': 'from_file',
 'repeat_year_forcing': False,
 'ocean_mask': None,
 'layout': None,
 'minimum_depth': 9.95,
 'tidal_constituents': ['M2',
  'S2',
  'N2',
  'K2',
  'K1',
  'O1',
  'P1',
  'Q1',
  'MM',
  'MF'],
 'boundaries': ['south', 'north', 'east'],
 'mom_input_dir': '/glade/u/home/manishrv/documents/nwa12_0.1/mom_input/gfdl_copy.2.n3b',
 'mom_run_dir': '/glade/u/home/manishrv/documents/nwa12_0.1/mom_run/gfdl_copy.2.n3b',
 'bathymetry': '/glade/u/home/manishrv/documents/nwa12_0.1/mom_input/gfdl_copy.2.n3b/bathymetry.nc',
 'hgrid': '/glade/u/home/manishrv/documents/nwa12_0.1/mom_input/gfdl_copy.2.n3b/hgrid.nc',
 'vgrid': '/glade/u/home/manishrv/documents/n

## Step 3: Prepare ocean forcing data

We need to cut out our ocean forcing. The package expects an initial condition and one time-dependent segment per non-land boundary. Naming convention is `"east_unprocessed"` for segments and `"ic_unprocessed"` for the initial condition.

In this notebook, we are forcing with the Copernicus Marine "Glorys" reanalysis dataset. There's a function in the `mom6-regional` package that generates a bash script to download the correct boundary forcing files for your experiment. First, you will need to create an account with Copernicus, and then call `copernicusmarine login` to set up your login details on your machine. Then you can run the `get_glorys_data.sh` bash script.

The function is called `get_glorys_rectangular` because the fully automated setup is only supported for domains with boundaries parallel to lines of longitude and latitude. To download more complex domain shapes you can call `rmom6.get_glorys_data` directly.

In [6]:
expt.get_glorys(
    raw_boundaries_path=glorys_path,
)

The script `get_glorys_data.sh` has been generated at:
  /glade/derecho/scratch/manishrv/gfdl_copy.4.n3b_inputs/glorys.
To download the data, run this script using `bash` in a terminal with internet access.

Important instructions:
1. You will need your Copernicus Marine username and password.
   If you do not have an account, you can create one here: 
   https://data.marine.copernicus.eu/register
2. You will be prompted to enter your Copernicus Marine credentials multiple times: once for each dataset.
3. Depending on the dataset size, the download process may take significant time and resources.
4. Thus, on certain systems, you may need to run this script as a batch job.



##  Step 5: Handle the ocean forcing - where the magic happens

This cuts out and interpolates the initial condition as well as all boundaries (unless you don't pass it boundaries).

The dictionary maps the MOM6 variable names to what they're called in your ocean input file. Notice how for GLORYS, the horizontal dimensions are `latitude` and `longitude`, vs `xh`, `yh`, `xq`, `yq` for MOM6. This is because for an 'A' grid type tracers share the grid with velocities so there's no difference.

If one of your segments is land, you can delete its string from the 'boundaries' list. You'll need to update MOM_input to reflect this though so it knows how many segments to look for, and their orientations.

In [10]:
# Define a mapping from the GLORYS variables and dimensions to the MOM6 ones
ocean_varnames = {"time": "time",
                  "yh": "latitude",
                  "xh": "longitude",
                  "zl": "depth",
                  "eta": "zos",
                  "u": "uo",
                  "v": "vo",
                  "tracers": {"salt": "so", "temp": "thetao"}
                  }

# Set up the initial condition
expt.setup_initial_condition(
    Path(glorys_path) / "ic_unprocessed.nc", # directory where the unprocessed initial condition is stored, as defined earlier
    ocean_varnames,
    arakawa_grid="A"
    )    

# Set up the four boundary conditions. Remember that in the glorys_path, we have four boundary files names north_unprocessed.nc etc. 
expt.setup_ocean_state_boundaries(
        Path(glorys_path),
        ocean_varnames,
        arakawa_grid = "A"
        )

INITIAL CONDITIONS
Regridding Velocities... Done.
Regridding Tracers... Done.
Regridding Free surface... Done.
Saving outputs... done setting up initial condition.
NOTE: the 'setup_run_directories' method does understand the less than four boundaries but be careful. Please check the MOM_input/override file carefully to reflect the number of boundaries you have, and their orientations. You should be able to find the relevant section in the MOM_input/override file by searching for 'segment_'. Ensure that the segment names match those in your inputdir/forcing folder
Processing south boundary...2024-12-27 14:00:06,340 - regional_mom6.regridding.coords - INFO - Creating coordinates of the boundary q/u/v points
2024-12-27 14:00:06,343 - regional_mom6.regridding.create_regridder - INFO - Creating Regridder
2024-12-27 14:00:06,789 - regional_mom6.regridding.fill_missing_data - INFO - Forward filling in missing data along z-dim
2024-12-27 14:00:06,820 - regional_mom6.regridding.add_or_update_ti

In [5]:
expt.setup_boundary_tides(Path("/glade/u/home/manishrv/manish_scratch_symlink/inputs_rm6/tidal_data/h_tpxo9.v1.nc"),Path("/glade/u/home/manishrv/manish_scratch_symlink/inputs_rm6/tidal_data/u_tpxo9.v1.nc"))

Processing south boundary...2024-12-27 10:32:34,519 - regional_mom6.regridding.coords - INFO - Creating coordinates of the boundary q/u/v points
2024-12-27 10:32:34,521 - regional_mom6.regridding.create_regridder - INFO - Creating Regridder
2024-12-27 10:32:59,511 - regional_mom6.regridding.fill_missing_data - INFO - Forward filling in missing data along z-dim
2024-12-27 10:32:59,514 - regional_mom6.regridding.fill_missing_data - INFO - Forward filling in missing data along z-dim
2024-12-27 10:32:59,519 - regional_mom6.regridding.add_or_update_time_dim - INFO - Adding time dimension
2024-12-27 10:32:59,535 - regional_mom6.regridding.coords - INFO - Creating coordinates of the boundary q/u/v points
2024-12-27 10:32:59,537 - regional_mom6.regridding.add_secondary_dimension - INFO - Adding perpendicular dimension to zamp_segment_001
2024-12-27 10:32:59,539 - regional_mom6.regridding.add_secondary_dimension - INFO - Adding perpendicular dimension to zphase_segment_001
2024-12-27 10:32:59,5

## Step 6: Modify the default input directory to make a (hopefully) runnable configuration out of the box

This step copies the default directory and modifies the `MOM_layout` files to match your experiment by inserting the right number of x, y points and CPU layout.

To run MOM6 using the [payu infrastructure](https://github.com/payu-org/payu), provide the keyword argument `using_payu = True` to the `setup_run_directory` method and an example `config.yaml` file will be appear in the run directory. The `config.yaml` file needs to be modified manually to add the locations of executables, etc.

In [12]:
expt.setup_run_directory(surface_forcing = "jra", with_tides = True)

Could not find premade run directories at  /glade/u/home/manishrv/documents/nwa12_0.1/regional_mom_workflows/CrocoDash/CrocoDash/rm6/regional_mom6/demos/premade_run_directories
Perhaps the package was imported directly rather than installed with conda. Checking if this is the case... 
Found run files. Continuing...
At least one of these is requiret to set up the experiment if you're running MOM6 standalone with the FMS coupler. 
If you're running within CESM, ignore this message.
Deleting indexed OBC keys from MOM_input_dict in case we have a different number of segments


In [14]:
expt.change_MOM_parameter("MAXIMUM_DEPTH",'7000', override = True) # Fred's Vertical Coordinate has a depth of 6500

This parameter MAXIMUM_DEPTH is being replaced from 7000 to 7000 in MOM_override


'7000'

# SECTION 2: Create a blank CESM run

So far I've used Alper's [GUI](https://github.com/ESMCI/visualCaseGen?tab=readme-ov-file) branch of CESM. Clone respective branch of CESM and then run the generate case command. Below is the command I used to generate a global, MOM6 only run forced with JRA data atmosphere

`/glade/u/home/manishrv/manish_scratch_symlink/installs/clean_cesm3_0_alpha03a/cime/scripts/create_newcase --compset 1850_DATM%JRA_SLND_SICE_MOM6_SROF_SGLC_SWAV --res TL319_t232 --case /glade/u/home/manishrv/cases/gfdl_copy.1.n3b --machine derecho --run-unsupported --project p93300012 --non-local`

In [13]:
# Save the path where the new CESM config lives
CESM_path = Path(f"/glade/u/home/manishrv/cases/gfdl_copy.1.n3b")

# SECTION 3: Modify the CESM run to make it regional

In [15]:
## Regional CaseGen is part of CrocoDash

reg_cas_obj = cd.regional_casegen.cesm_tools.RegionalCaseGen()


reg_cas_obj.setup_cesm(expt, CESM_path, project = "P93300012")
## OR you can explicitly state what variables are needed
# reg_cas_obj.setup_cesm_explicit(hgrid = expt.hgrid, CESMPath = CESM_path,project = "p93300612", mom_input_dir=expt.mom_input_dir, mom_run_dir=expt.mom_run_dir,date_range = expt.date_range)

2024-12-27 13:02:07,699 - CrocoDash.regional_casegen.cesm_tools - INFO - Copying input.nml, diag_table, MOM_input_and MOM_override to /glade/u/home/manishrv/cases/gfdl_copy.1.n3b/SourceMods/src.mom
2024-12-27 13:02:07,737 - CrocoDash.regional_casegen.cesm_tools - INFO - Adding NIGLOBAL = 740, NJGLOBAL = 780, and INPUTDIR = /glade/u/home/manishrv/documents/nwa12_0.1/mom_input/gfdl_copy.1.n3b to MOM_override
2024-12-27 13:02:07,738 - CrocoDash.regional_casegen.cesm_tools - INFO - Add MOM_override to parameter_filename in input.nml
2024-12-27 13:02:07,739 - CrocoDash.regional_casegen.cesm_tools - INFO - Removing reference to MOM_layout in input.nml and add MOM_override to parameter_filename in input.nml
2024-12-27 13:02:07,740 - CrocoDash.regional_casegen.cesm_tools - INFO - 
2024-12-27 13:02:07,741 - CrocoDash.regional_casegen.cesm_tools - INFO - Modifying line: parameter_filename = 'MOM_input', 'MOM_layout', 'MOM_override'
2024-12-27 13:02:07,741 - CrocoDash.regional_casegen.cesm_tools 

Now hopefully doing the usual 

`./case.setup && ./case.build && ./case.submit` 

should at least run. Of course from here you'll have a lot of other things to fiddle around with to make it run *well!* 